# Index Rebalance Tracker — Methodology Walkthrough

This notebook walks through the full analytical pipeline end-to-end on the live SP500 data set. It is intended as the **publication companion** to the project's README and to `docs/METHODOLOGY.md` (when generated).

Run order:
1. **Data** — load constituents + historical changes from `data/`, fetch the price universe via the cached `PriceCache`.
2. **Event study** — market-model AR (Brown & Warner 1985) and sector-matched controls.
3. **Liquidity** — Amihud (2002), Corwin-Schultz (2012), Kyle (1985 daily-data variant).
4. **TCA** — square-root impact (Almgren et al. 2005) for forced vs spread execution of a hypothetical \$6.5T passive fund.
5. **Decay** — annual cohort means with IQR. **Headline result.**

Charts use `matplotlib` with the project's dark palette.

In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from index_rebalance_tracker.analysis.decay import cohort_decay
from index_rebalance_tracker.analysis.event_study import run_event_study
from index_rebalance_tracker.analysis.liquidity import (
    amihud_illiquidity,
    corwin_schultz_spread,
    daily_return_vol,
    kyle_lambda,
)
from index_rebalance_tracker.analysis.sector_match import adv_dollars_60d
from index_rebalance_tracker.analysis.tca import (
    estimated_demand_usd,
    implementation_shortfall_summary,
)
from index_rebalance_tracker.data.prices import PriceCache
from index_rebalance_tracker.models import IndexEvent

# Project palette (matches the dashboard)
TEAL, ROSE, AMBER, ZINC = "#14B8A6", "#F43F5E", "#FACC15", "#A1A1AA"
plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#18181b"
plt.rcParams["axes.facecolor"] = "#18181b"
plt.rcParams["savefig.facecolor"] = "#18181b"
plt.rcParams["axes.edgecolor"] = "#3f3f46"
plt.rcParams["axes.labelcolor"] = "#e4e4e7"
plt.rcParams["text.color"] = "#e4e4e7"
plt.rcParams["xtick.color"] = "#a1a1aa"
plt.rcParams["ytick.color"] = "#a1a1aa"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#27272a"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")

## 1 · Load M1 outputs and the price universe

Run `index-rebalance pull-history --index sp500 --start 2010-01-01` from the repo root before this cell.

In [ ]:
events_df = pd.read_parquet(DATA_DIR / "events_sp500.parquet")
cons_df = pd.read_parquet(DATA_DIR / "constituents_sp500.parquet")
events_df = events_df[events_df["effective_date"] >= date(2010, 1, 1)]
events = [IndexEvent.model_validate(r) for r in events_df.to_dict("records")]
events_by_id = {e.event_id: e for e in events}
print(f"{len(events)} events 2010-{events_df['effective_date'].max().year}")
events_df["effective_date"].dt.year.value_counts().sort_index()

In [ ]:
cache = PriceCache()
universe = sorted({e.ticker for e in events} | set(cons_df["ticker"]))
price_start = (events_df["effective_date"].min() - pd.Timedelta(days=400)).date()
price_end = (events_df["effective_date"].max() + pd.Timedelta(days=60)).date()

market_df = cache.fetch_prices("SPY", price_start, price_end)
market_returns = market_df["adj_close"].pct_change()
market_returns.index = pd.to_datetime(market_returns.index)

returns, prices = {}, {}
for t in universe:
    try:
        df = cache.fetch_prices(t, price_start, price_end)
    except Exception:
        continue
    if df.empty:
        continue
    prices[t] = df
    s = df["adj_close"].pct_change()
    s.index = pd.to_datetime(s.index)
    returns[t] = s
print(f"fetched {len(prices)} tickers")
adv_map = adv_dollars_60d(prices)
sector_map = dict(zip(cons_df["ticker"], cons_df["gics_sector"]))

## 2 · Event study — market model + sector-matched

Each event produces up to 4 windows × 2 models = 8 CAR rows. We run both methods so the headline chart can compare their confidence intervals.

In [ ]:
car_obs = run_event_study(
    events=events,
    returns=returns,
    market_returns=market_returns,
    sector_map=sector_map,
    adv_dollars_60d=adv_map,
    model="both",
    pre_run_up_days=5,
    post_drift_days=20,
)
car_df = pd.DataFrame([o.model_dump() for o in car_obs])
print(f"{len(car_df)} CAR rows; {car_df['window_label'].value_counts().to_dict()}")
print(car_df.groupby(["model", "window_label"])["car"].describe()[["count", "mean", "std"]])

## 3 · Headline chart — index-effect decay over time

In [ ]:
cohorts = cohort_decay(car_obs, events_by_id, grouping="yearly")
decay_df = pd.DataFrame([c.model_dump() for c in cohorts])
decay_df["cohort"] = decay_df["cohort"].astype(int)
decay_df = decay_df.sort_values("cohort")
decay_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(
    decay_df["cohort"], decay_df["car_p25"], decay_df["car_p75"], color=TEAL, alpha=0.2, label="IQR"
)
ax.plot(
    decay_df["cohort"],
    decay_df["mean_car"],
    "o-",
    color=TEAL,
    markersize=8,
    linewidth=2,
    label="mean CAR (additions)",
)
ax.axhline(0, color=ZINC, linestyle=":", linewidth=1)
ax.set_title(
    "SP500 addition CAR by year cohort  ([T-A, T-E-1] window, market model)", pad=12, fontsize=13
)
ax.set_ylabel("cumulative abnormal return")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0, decimals=1))
ax.set_xlabel("cohort year")
ax.legend(loc="upper right", frameon=False)
plt.tight_layout()
plt.savefig("../assets/decay_headline.png", dpi=144, bbox_inches="tight")
plt.show()

## 4 · Liquidity diagnostics across events

In [ ]:
liq_rows = []
for e in events:
    if e.ticker not in prices:
        continue
    df = prices[e.ticker]
    liq_rows.append(
        {
            "event_id": e.event_id,
            "ticker": e.ticker,
            "year": e.effective_date.year,
            "adv_dollars": adv_map.get(e.ticker, np.nan),
            "cs_spread": corwin_schultz_spread(df),
            "amihud": amihud_illiquidity(df),
            "kyle": kyle_lambda(df, window=60),
            "daily_vol": daily_return_vol(df, window=60),
        }
    )
liq_df = pd.DataFrame(liq_rows)
liq_df.describe()[["adv_dollars", "cs_spread", "amihud", "daily_vol"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(liq_df["cs_spread"] * 1e4).dropna().plot.hist(ax=axes[0], bins=30, color=TEAL, edgecolor="#27272a")
axes[0].set_title("Corwin-Schultz spread proxy")
axes[0].set_xlabel("spread (bps)")
(liq_df["daily_vol"] * 100).dropna().plot.hist(ax=axes[1], bins=30, color=ROSE, edgecolor="#27272a")
axes[1].set_title("60d daily-return vol")
axes[1].set_xlabel("vol (%)")
plt.tight_layout()
plt.show()

## 5 · TCA — implementation shortfall for hypothetical $6.5T passive fund

In [ ]:
passive_aum = 6.5e12
total_adv = sum(adv_map.values())
tca_rows = []
for e in events:
    if e.ticker not in prices:
        continue
    adv_d = adv_map.get(e.ticker, 0.0)
    if adv_d <= 0:
        continue
    demand = estimated_demand_usd(passive_aum, adv_d, total_adv)
    vol = daily_return_vol(prices[e.ticker], window=60)
    s = implementation_shortfall_summary(demand, adv_d, vol, spread_n_days=5)
    tca_rows.append(
        {
            "year": e.effective_date.year,
            "ticker": e.ticker,
            "forced_bps": s["forced_bps"],
            "spread_bps": s["spread_bps"],
            "savings_bps": s["savings_bps"],
        }
    )
tca_df = pd.DataFrame(tca_rows).dropna()
annual = tca_df.groupby("year")[["forced_bps", "spread_bps", "savings_bps"]].mean()
annual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(annual.index, annual["forced_bps"], "o-", color=ROSE, label="forced T-E execution")
ax.plot(annual.index, annual["spread_bps"], "o-", color=TEAL, label="5-day spread execution")
ax.fill_between(
    annual.index,
    annual["spread_bps"],
    annual["forced_bps"],
    color=AMBER,
    alpha=0.15,
    label="savings (bps)",
)
ax.set_title("Implementation shortfall — forced vs spread execution", fontsize=13, pad=12)
ax.set_ylabel("cost (bps)")
ax.set_xlabel("year")
ax.legend(loc="upper right", frameon=False)
plt.tight_layout()
plt.savefig("../assets/tca_headline.png", dpi=144, bbox_inches="tight")
plt.show()

## 6 · Honest takeaways

* **Sample size**: SP500 typically has 20–30 changes per year; cohort-level confidence intervals are wide. Bootstrap or robust SE recommended for any production claim.
* **GICS sector drift**: ~5–10% of stocks change sector classification over 15 years. Sector-matched controls are slightly biased for the oldest events.
* **Float-weight proxy**: ADV-share-of-index is highly correlated with float-weight within sector, but real index providers compute weights via float-adjusted shares × price. This module documents the limit.
* **Daily-data limits**: Amihud, Corwin-Schultz, and Kyle's lambda are all proxies. True intraday TCA needs trade-by-trade data we don't pay for.

These are stated explicitly so a recruiter / hiring manager can verify the author understands the boundary between rigorous and approximate.